### Test differential operators

Here, I'll try to test the implementation of differential operators by checking if they are actually zero or non-negative on optimal constructions.

- Goodman's bound
$$
  t(\triangle, G) \ge 2 t(e, G)^2 - t(e, G).
$$
where the optimal constructions are known as balanced complete $k$-partite graphons.

In [ ]:
import flag_algebra.flag_algebra as fa
import networkx as nx
import sympy

In [ ]:
import numpy as np
import math

def graph_density(H, k):
  poly = nx.chromatic_polynomial(H)

  x = sympy.symbols('x')

  return int(poly.subs(x, k)) / (k ** nx.number_of_nodes(H))

def falling_factorial(x, m):
  """
  Computes the falling factorial x^(m) = x * (x-1) * ... * (x-m+1)
  """
  res = 1
  for i in range(m):
    res *= (x - i)
  return res

def induced_graph_density(H, k):
  """
  Computes the induced homomorphism density of H in a balanced complete
  k-partite graphon.
  
  Returns 0 if H is not a complete multipartite graph.
  """
  n = H.number_of_nodes()
  H_bar = nx.complement(H)
  
  # Structural Check:
  # For induced density to be > 0, non-adjacency must be transitive.
  # This implies H_bar must be a disjoint union of cliques (cluster graph).
  components = list(nx.connected_components(H_bar))
  m = len(components) # This is the number of parts in H
  
  for comp in components:
    # Verify if this component induces a clique in H_bar
    subgraph = H_bar.subgraph(comp)
    num_nodes = len(comp)
    num_edges = subgraph.number_of_edges()
    
    # A clique of size N must have N*(N-1)/2 edges
    if num_edges != num_nodes * (num_nodes - 1) // 2:
      return 0
      
  # If valid, result is P(k, m) / k^n
  # We use sympy.ff (falling factorial) to handle symbolic k naturally
  return falling_factorial(k, m) / (k ** n)

e = nx.from_edgelist([(0, 1)])  # Edge graph
print("Density of edge in Kk:", graph_density(e, 3))

In [ ]:
# Goodman bound: f(K3, e) = K3 - 2 e^2 + e
# Its partial derivative 
# partial f / partial K3 = 1
# partial f / partial e = - 4 e + 1
# Therefore, the grad term is 
# Grad_{G,a}(f) = K3 + (1 - 4p) e

k3 = nx.from_edgelist([(0, 1), (1, 2), (2, 0)])  # Use t(K3, G).
edge = nx.from_edgelist([(0, 1)])              # Use t(e, G).
e2 = nx.from_edgelist([(0, 1), (2, 3)])        # Use t(e U e, G) = t(e, G)^2.

objective = [('sub', k3, 1), ('sub', e2, -2), ('sub', edge, 1)]
atlas = fa.get_graph_atlas(7)

grad_term = fa.vertex_differential(objective, atlas)

In [ ]:
print(grad_term.shape)

for k in [3, 4, 5, 6, 7]:
  densities = np.zeros(len(atlas))
  for i, G in enumerate(atlas):
    densities[i] = induced_graph_density(G, k)
  
  # res_j =density_i grad_term_ij
  res = np.dot(densities, grad_term)
  print(res)
  # Should be all zero.

This time, we test on induced density maximisation 
$$
  \max_G t_{\text{ind}}(C_5, G)
$$
whose optimal solution is known as iterative blow-up of $C_5$.

In [ ]:
import itertools
# This code computes induced density of H in iterative blow-up of G.
def calculate_induced_density(H, G):
  m = len(H)
  k = len(G)

  densities = {}

  all_subsets = []
  for r in range(1, m+1):
    all_subsets.extend(itertools.combinations(range(m), r))
  
  for subset_tuple in all_subsets:
    subset = frozenset(subset_tuple)
    size = len(subset)

    if size == 1:
      densities[subset] = 1.0
      continue

    sum_split_val = 0.0
    ordered_subset = sorted(list(subset))
    
    for mapping in itertools.product(range(k), repeat=size):
      unique_buckets = set(mapping)
      
      if len(unique_buckets) == 1:
        continue
      
      consistent = True
      
      for i in range(size):
        for j in range(i + 1, size):
          u_idx = ordered_subset[i]
          v_idx = ordered_subset[j]
          
          bucket_u_idx = mapping[i]
          bucket_v_idx = mapping[j]
          
          if bucket_u_idx != bucket_v_idx:
            is_edge_H = H.has_edge(u_idx, v_idx)
            is_edge_G = G.has_edge(bucket_u_idx, bucket_v_idx)
            
            if is_edge_H != is_edge_G:
              consistent = False
              break
        if not consistent:
          break
      
      if consistent:
        term = 1.0
        
        bucket_groups = {b: [] for b in unique_buckets}
        for i, bucket_idx in enumerate(mapping):
          bucket_groups[bucket_idx].append(ordered_subset[i])
        
        for b_nodes in bucket_groups.values():
          if len(b_nodes) > 0:
            term *= densities[frozenset(b_nodes)]
        
        sum_split_val += term

    densities[subset] = sum_split_val / (k**size - k)

  prob_density = densities[frozenset(range(m))]
  
  return prob_density

density = calculate_induced_density(nx.cycle_graph(5), nx.cycle_graph(5))

In [ ]:
objective = [('ind', nx.cycle_graph(5), -1)] # Use -1 to consider the minimisation problem

atlas = fa.get_graph_atlas(8)
grad_term = fa.vertex_differential(objective, atlas)

In [ ]:
import tqdm 

print(grad_term.shape)

densities = np.zeros(len(atlas))
for i, G in tqdm.tqdm(enumerate(atlas), total=len(atlas)):
  densities[i] = calculate_induced_density(G, nx.cycle_graph(5))
  
res = np.dot(densities, grad_term)
print(res)